In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver#to store state in RAM rather than erasing it when workflow reaches end--called as persistance

load_dotenv()

class ChatState(TypedDict):
    
    messages: Annotated[list[BaseMessage], add_messages]
    #all other msgs such as human message,ai msg,system msg..inherits Basemessage

llm = ChatGoogleGenerativeAI(model='gemini-3.6-flash')

In [4]:
def chat_node(state:ChatState):
    messages=state['messages']
    response = llm.invoke(messages)
    return{'messages':[response]}


In [5]:
checkpointer=MemorySaver()

graph=StateGraph(ChatState)
graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot = graph.compile(checkpointer)

In [6]:

# while True:
#     user_message = input('Type here:')
#     print('User: ',user_message)
#     if user_message.strip().lower() in ['exit','quit','stop','bye']:
#         break
#     response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]})
  
#     print('AI:', response['messages'][-1].content)

#before adding memory->persistence--llm dont remember prev msgs..because every time loop run it starts from start node and ends at the END node
"""curr_state={'messages':[]}
while True:
    user_message = input('Type here:')
    print('User: ',user_message)
    if user_message.strip().lower() in ['exit','quit','stop','bye']:
        break
    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]})
  
    print('AI:', response['messages'][-1].content)

"""

"curr_state={'messages':[]}\nwhile True:\n    user_message = input('Type here:')\n    print('User: ',user_message)\n    if user_message.strip().lower() in ['exit','quit','stop','bye']:\n        break\n    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]})\n\n    print('AI:', response['messages'][-1].content)\n\n"

In [ ]:
#implementation of short term memory through persistence

while True:
    user_message = input('Type here:')
    print('User: ',user_message)
    if user_message.strip().lower() in ['exit','quit','stop','bye']:
        break

    config={'configurable':{'thread_id':'1'}}
    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)
  
    print('AI:', response['messages'][-1].content)

User:  hi
AI: [{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EvYECvMEARFNMg/fPpqtlb4XmID8w46gaccpsoGhnYrf4eYE8WjxrQJhg/dyawNua9X+3G6PMOZHlBLuRK+8Pb/HDXITky16euKGP15XxHUwZXuOTz+qKrvQpPFNG9gJo0CV0vrmFDuzGUnuTEpJfQfIrU2wCP3dWgcAY537/A93UX/GlbPS76TUdg7Qc1YMYGv3vz8coNRlYehr1h7cocOs6+5Bp2imgXSSd1LYuufBOA2NefkVUNoEcSad5uF8g3Wgushqh4V8D2o4GkUfxEjVfOvZQTmbpoDJXZTKPxRIAJZPI9G7iedCozo9c+pDaCSKleLWvMSa8cd2WJ1Y1B10QLUK5AMWqBem1rOUeuKsBuMyMLPLE7z/9jpT0eOZn8iXL515v0x3dmdU8LAzi0JvET8CXv+0OqwKJZu38VDHx5yTFzqtnlIvKTxf7rSXlqB4YcimTsd78kY3CJotF2QrZy7bnc7zHuKDT+LqhpBs9pGivLyNgrqm+9GCYUEQmpbPj6AewFtcTC+zKoBAQqEe0RgeFxjpfOaPnrM7xiQx4mAkjfpJNjikcOmXclVMdF8izQz1M647AhY/+fzc4cEmRx4vUnVWdDyjdk1u2I+5+rBWSuSubjvI3To6vNtTBuXswiV0UyGKlbSxGjyF8MJDeIePCcGUl/lTF9k1PL9SAdCZAZ57TOzZ5AoYy3XyaW3SrckJXeby3qff4Elm4Q72LmHX+lhzCCS8J/d6bCTYEX9dPKp1nIzu8ObEC2MIE+Hrl/FcbsOlzDdGrzeSjgqu/WY3ulHRfyt94UVXfZ8TGBHYTpEuIgmFw6gCtvGMHAfTTIuL1b+9'}}]
User:  bye


In [1]:
#without checkpointer-persistence-history wont be stored..each msg-new workflow
# # res = chatbot.invoke({'messages': [HumanMessage(content="hi")]})

# # Extract the text answer content from the last message in the returned list
# result_text = res['messages']
# print(result_text)


In [9]:
chatbot.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='0d6eb1b8-6428-44b2-b21c-412ff0170f2b'), AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EvYECvMEARFNMg/fPpqtlb4XmID8w46gaccpsoGhnYrf4eYE8WjxrQJhg/dyawNua9X+3G6PMOZHlBLuRK+8Pb/HDXITky16euKGP15XxHUwZXuOTz+qKrvQpPFNG9gJo0CV0vrmFDuzGUnuTEpJfQfIrU2wCP3dWgcAY537/A93UX/GlbPS76TUdg7Qc1YMYGv3vz8coNRlYehr1h7cocOs6+5Bp2imgXSSd1LYuufBOA2NefkVUNoEcSad5uF8g3Wgushqh4V8D2o4GkUfxEjVfOvZQTmbpoDJXZTKPxRIAJZPI9G7iedCozo9c+pDaCSKleLWvMSa8cd2WJ1Y1B10QLUK5AMWqBem1rOUeuKsBuMyMLPLE7z/9jpT0eOZn8iXL515v0x3dmdU8LAzi0JvET8CXv+0OqwKJZu38VDHx5yTFzqtnlIvKTxf7rSXlqB4YcimTsd78kY3CJotF2QrZy7bnc7zHuKDT+LqhpBs9pGivLyNgrqm+9GCYUEQmpbPj6AewFtcTC+zKoBAQqEe0RgeFxjpfOaPnrM7xiQx4mAkjfpJNjikcOmXclVMdF8izQz1M647AhY/+fzc4cEmRx4vUnVWdDyjdk1u2I+5+rBWSuSubjvI3To6vNtTBuXswiV0UyGKlbSxGjyF8MJDeIePCcGUl/lTF9k1PL9SAdCZAZ57TOzZ5AoYy3XyaW3SrckJXeby3qff4Elm4Q72LmHX+lhzCCS8J/d6b

In [12]:
x=chatbot.get_state_history(config)
list(x)

[StateSnapshot(values={'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='0d6eb1b8-6428-44b2-b21c-412ff0170f2b'), AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EvYECvMEARFNMg/fPpqtlb4XmID8w46gaccpsoGhnYrf4eYE8WjxrQJhg/dyawNua9X+3G6PMOZHlBLuRK+8Pb/HDXITky16euKGP15XxHUwZXuOTz+qKrvQpPFNG9gJo0CV0vrmFDuzGUnuTEpJfQfIrU2wCP3dWgcAY537/A93UX/GlbPS76TUdg7Qc1YMYGv3vz8coNRlYehr1h7cocOs6+5Bp2imgXSSd1LYuufBOA2NefkVUNoEcSad5uF8g3Wgushqh4V8D2o4GkUfxEjVfOvZQTmbpoDJXZTKPxRIAJZPI9G7iedCozo9c+pDaCSKleLWvMSa8cd2WJ1Y1B10QLUK5AMWqBem1rOUeuKsBuMyMLPLE7z/9jpT0eOZn8iXL515v0x3dmdU8LAzi0JvET8CXv+0OqwKJZu38VDHx5yTFzqtnlIvKTxf7rSXlqB4YcimTsd78kY3CJotF2QrZy7bnc7zHuKDT+LqhpBs9pGivLyNgrqm+9GCYUEQmpbPj6AewFtcTC+zKoBAQqEe0RgeFxjpfOaPnrM7xiQx4mAkjfpJNjikcOmXclVMdF8izQz1M647AhY/+fzc4cEmRx4vUnVWdDyjdk1u2I+5+rBWSuSubjvI3To6vNtTBuXswiV0UyGKlbSxGjyF8MJDeIePCcGUl/lTF9k1PL9SAdCZAZ57TOzZ5AoYy3XyaW3SrckJXeby3qff4Elm4Q72LmHX+lhzCCS8J/d6